# Tutorial 2: Plotting and CLI options

This tutorial expands on {doc}`Tutorial 1 <tutorial_1>` with visualization and demonstrating the behavior of some key CLI options.

This tutorial assumes you have `floodsr` installed per {doc}`Tutorial 1 <tutorial_1>`. 


 


# Install additional packages

For this tutorial, we need the additional python packages `matplotlib` and `rasterio`. Like the install of `floodsr` shown in {doc}`Tutorial 1 <tutorial_1>`, obtaining and implementing these packages depends on your execution context:

- **command line (CLI)**: assuming `floodsr` was installed in an isoplated environment via pipx, you'll need a separate environment to run the non-CLI commands shown in this tutorial. For this, you can use an existing Python environment, or create a new one with conda or venv, then install `matplotlib` and `rasterio` there. You may need to re-install `floodsr` via pipx into your new environment. Don't forget to remove the `!` prefix when copying and pasting floodsr commands from this notebook into your terminal.
- **local notebook (Jupyter)**: if needed, uncomment the below to patch `matplotlib` and `rasterio` into your active kernel environment. then re-launch this notebook or switch the kernel to that environment.
- **hosted notebook (Colab)**: Colab should have `matplotlib` and `rasterio` pre-installed.

In [ ]:
# %pip install -q matplotlib rasterio

Let's check the versions now:

In [ ]:
import matplotlib
import rasterio

print(f"matplotlib=={matplotlib.__version__}")
print(f"rasterio=={rasterio.__version__}")

## Imports


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
from rasterio.warp import Resampling, calculate_default_transform, reproject


In [ ]:
# print cwd
print(f"Current working directory: {Path.cwd()}")

## Download the same test data as Tutorial 1

If you already downloaded these files while working through Tutorial 1, you can skip this. 


In [ ]:
urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/hires002_dem.tif",
    "hires002_dem.tif",
)
urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/lowres032.tif",
    "lowres032.tif",
)
lowres_fp = Path("lowres032.tif").resolve()
dem_fp = Path("hires002_dem.tif").resolve()
assert lowres_fp.is_file(), f"missing low-res raster\n    {lowres_fp}"
assert dem_fp.is_file(), f"missing DEM raster\n    {dem_fp}"


## Plot the tutorial inputs

We start by plotting the low-resolution flood-depth raster in blue and the DEM tile with a terrain colormap.


In [ ]:
with rasterio.open(lowres_fp) as src:
    lowres_arr = src.read(1).astype(float)
    if src.nodata is not None:
        lowres_arr[lowres_arr == src.nodata] = np.nan

with rasterio.open(dem_fp) as src:
    dem_arr = src.read(1).astype(float)
    if src.nodata is not None:
        dem_arr[dem_arr == src.nodata] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(np.where(lowres_arr > 0, lowres_arr, np.nan), cmap="Blues")
axes[0].set_title("Low-res flood depth")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
im1 = axes[1].imshow(dem_arr, cmap="terrain")
axes[1].set_title("High-res DEM")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()


this shows the not-so-great pluvial flood raster at 32m

## Probe the CRS of both rasters

A coordinate reference system (CRS) tells raster software how pixel locations map onto real-world coordinates.

Before we trigger a CRS mismatch on purpose, let's inspect the CRS metadata on the original rasters.


In [ ]:
with rasterio.open(lowres_fp) as src:
    print(f"{lowres_fp.name}: crs={src.crs} shape={src.shape} res={src.res}")

with rasterio.open(dem_fp) as src:
    print(f"{dem_fp.name}: crs={src.crs} shape={src.shape} res={src.res}")


Here we see that the CRS of both rasters are identical. 

Nice for simple test data, but not so interesting for demonstrating how `floodsr` handles CRS mismatch.


## Create a CRS mismatch on purpose

To demonstrate `--crs-policy`, we reproject the low-resolution raster to `3978` and save it as a new GeoTIFF.


In [ ]:
!rio warp lowres032.tif lowres032_epsg3978.tif --dst-crs EPSG:3978 --resampling nearest --co TILED=YES

## Run `tohr` with the mismatched CRS

This command should fail because the low-resolution raster and the DEM no longer share the same CRS and we haven't told `floodsr` how to handle this mismatch yet, so it assumes `--crs-policy strict` by default. 


In [ ]:
!floodsr tohr --in lowres032_epsg3978.tif --dem hires002_dem.tif


## What `--crs-policy` does

`--crs-policy` controls how `floodsr` handles CRS mismatches between the low-resolution depth raster and the DEM. This is necessary because `tohr` needs the rasters to line up on a common projected grid before preprocessing and inference can proceed, and there is not an obvious "right answer" for which CRS to use when they don't match.

For this example we use `--crs-policy=use-dem`, which tells `floodsr` to treat the DEM CRS as the target CRS and reproject the low-resolution depth raster to match it.
We could have chosen `--crs-policy=use-lores` which would treat the low-resolution raster CRS as the target and reproject the DEM to match it.


In [ ]:
!floodsr tohr --in lowres032_epsg3978.tif --dem hires002_dem.tif --crs-policy=use-dem --out lowres032_epsg3978_use_dem_sr.tif


## Plot the CRS-policy result

let's visualize the result


In [ ]:
use_dem_fp = Path("lowres032_epsg3978_use_dem_sr.tif").resolve()
with rasterio.open(use_dem_fp) as src:
    use_dem_arr = src.read(1).astype(float)
    if src.nodata is not None:
        use_dem_arr[use_dem_arr == src.nodata] = np.nan

plt.figure(figsize=(5, 4))
plt.imshow(np.where(use_dem_arr > 0, use_dem_arr, np.nan), cmap="Blues")
plt.title("`--crs-policy=use-dem` result")
plt.axis("off")
plt.tight_layout()


## `--min-depth-threshold`

Now lets explore the argument `--min-depth-threshold`, which sets the minimum predicted depth retained in the output raster. 
Values below the threshold are written as `0.0`.

This is useful because very small positive predictions can be visually noisy or physically unimportant, and a user may want to tune this threshold to produce a drier or wetter result depending on their use case.

First, let's run `tohr` with the default `--min-depth-threshold` (which is 0.01) against the original CRS matching rasters as a comparison.


In [ ]:
!floodsr tohr --in lowres032.tif --dem hires002_dem.tif --out lowres032_default_sr.tif -q


In [ ]:
!floodsr tohr --in lowres032.tif --dem hires002_dem.tif --min-depth-threshold=0.1 --out lowres032_min_depth_01_sr.tif


## Compare the default threshold and `0.1`

The labels below show the percent of wet pixels in each raster, where `0` is treated as dry.


In [ ]:
default_fp = Path("lowres032_default_sr.tif").resolve()
threshold_fp = Path("lowres032_min_depth_01_sr.tif").resolve()

with rasterio.open(default_fp) as src:
    default_arr = src.read(1).astype(float)
    if src.nodata is not None:
        default_arr[default_arr == src.nodata] = np.nan

with rasterio.open(threshold_fp) as src:
    threshold_arr = src.read(1).astype(float)
    if src.nodata is not None:
        threshold_arr[threshold_arr == src.nodata] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plot_l = [
    (axes[0], np.where(lowres_arr > 0, lowres_arr, np.nan), "Input low-res"),
    (axes[1], np.where(default_arr > 0, default_arr, np.nan), "Default threshold"),
    (axes[2], np.where(threshold_arr > 0, threshold_arr, np.nan), "`--min-depth-threshold=0.1`"),
]
for ax, arr, title in plot_l:
    wet_pct = 100.0 * float(np.count_nonzero(np.nan_to_num(arr, nan=0.0) > 0.0)) / float(arr.size)
    ax.imshow(arr, cmap="Blues")
    ax.set_title(title)
    ax.axis("off")
    ax.text(0.03, 0.03, f"wet={wet_pct:.1f}%", transform=ax.transAxes, color="black", fontsize=10, bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"})
fig.tight_layout()


## `--window-method=hard`

`--window-method` controls how tiled model outputs are stitched together. The default `feather` mode blends overlapping windows, while `hard` uses non-overlapping tiles with direct writes.

For a small tile like this one there may be little visible difference, but `hard` is still useful to demonstrate because it changes how tiled inference is mosaicked.


In [ ]:
!floodsr tohr --in lowres032.tif --dem hires002_dem.tif --window-method=hard --tile-overlap=0 --out lowres032_hard_sr.tif


## Plot the hard-window result


In [ ]:
hard_fp = Path("lowres032_hard_sr.tif").resolve()
with rasterio.open(hard_fp) as src:
    hard_arr = src.read(1).astype(float)
    if src.nodata is not None:
        hard_arr[hard_arr == src.nodata] = np.nan

plt.figure(figsize=(5, 4))
plt.imshow(np.where(hard_arr > 0, hard_arr, np.nan), cmap="Blues")
plt.title("`--window-method=hard` result")
plt.axis("off")
plt.tight_layout()


## Next steps

This tutorial introduced plotting, CRS mismatch handling, and a few CLI knobs that affect the output. Continue with Tutorial 3 for the larger-raster workflow.
